# Numerical Quadrature

This page describes how `smudgy` integrates particle kernels over grid cells when depositing particle quantities onto a grid, and the options available to control it.

For a cell and the set $\mathcal{N}$ of particles whose kernels overlap it, deposition computes

$$
f_\mathrm{cell} = \sum_{i\in\mathcal N} m_i\, f_i \, I_{\mathrm{cell},i},
\qquad
I_{\mathrm{cell},i} = \int_{\Omega_\mathrm{cell}} W(\mathbfit{x}-\mathbfit{x}_i,\mathbfss{H}_i) \, \mathrm d\mathbfit{x},
$$

where $I_{\mathrm{cell},i}$ is the fraction of the kernel weight of particle $i$ within the cell domain $\Omega_\mathrm{cell}$, and $\mathbfss{H}_i$ denotes either the smoothing length $h_i$ or the smoothing tensor. For the fixed-grid B-splines and the `separable` kernels, the cell integral factorizes into one-dimensional integrals, which are evaluated exactly from the antiderivatives at the cell boundaries. The radial `isotropic` and ellipsoidal `covariant` kernels, in contrast, are not separable in Cartesian coordinates, and $I_{\mathrm{cell},i}$ must be evaluated numerically.

`smudgy` focuses on three aspects of this numerical integration: **accuracy**, **conservation** of particle mass (or weight) and **anti-aliasing**.

## Accuracy

If the kernel spans several cells, each cell integral is approximated by a tensor-product quadrature rule,

$$
I_{\mathrm{cell},i} \approx V_\mathrm{cell} \sum_{j} \omega_j\, W(\mathbfit{x}_j-\mathbfit{x}_i, \mathbfss{H}_i), \qquad \sum_j \omega_j = 1,
$$

where $V_\mathrm{cell}$ is the cell volume and $\mathbfit{x}_j$ are the quadrature nodes with weights $\omega_j$. The rule is selected with the `integration_method` argument of {py:meth}`~smudgy.pointcloud.PointCloud.deposit`. In one dimension, for a cell of width $s$ centred on $x_c$:

+ **`midpoint`** (default): one node at the cell centre. The fastest option.

   $$ I \approx s\, W(x_c) $$

+ **`trapezoidal`**: the $2^d$ equally weighted cell corners.

   $$ I \approx \frac{s}{2} \left[ W(x_c - s/2) + W(x_c + s/2) \right] $$

+ **`simpson`**: the $3^d$ nodes of Simpson's rule, which approximates the integrand with a quadratic polynomial.

   $$ I \approx \frac{s}{6} \left[ W(x_c - s/2) + 4W(x_c) + W(x_c + s/2) \right] $$

The midpoint and trapezoidal rules have local errors of $\mathcal{O}(V_\mathrm{cell}^{2/d})$, the midpoint rule with roughly half the error constant of the trapezoidal rule, whereas Simpson's rule reaches $\mathcal{O}(V_\mathrm{cell}^{4/d})$ at the cost of more kernel evaluations per cell. For covariant kernels, the overlapping cells are identified from the axis-aligned bounding box of the support ellipsoid, and the kernel is evaluated in the principal frame of $\mathbfss{H}_i$.

The figure below illustrates the three rules in one dimension:

```{image} ../src/kernel_quadrature_methods.png
:alt: Quadrature methods
:width: 60%
:align: center
```

## Conservation of mass

Since quadrature is only approximate, a particle's cell integrals generally do not sum exactly to the kernel normalization

$$
\int W(\mathbfit{x}-\mathbfit{x}_i,\, \mathbfss{H}_i) \,\mathrm{d}\mathbfit{x} = 1.
$$

Any deviation would mean that the total mass (or weight) of the particles is not conserved by the deposition. To guarantee exact conservation regardless of quadrature error, `smudgy` rescales each particle's cell integrals by the inverse of their sum:

```python
# for every particle p
total_integral = 0.0
for cell in support_cells:
    cell.integral = quadrature(p, cell)
    total_integral += cell.integral

rescaling_factor = 1.0 / total_integral
for cell in support_cells:
    cell.integral *= rescaling_factor
```

In practice, both passes are fused into a single deposition kernel that recomputes the inexpensive cell integrals rather than storing them. This rescaling may slightly distort the relative contributions of individual cells, but it conserves mass exactly. The sample-table path described below needs no correction, since the table is normalized by construction.

(anti-aliasing)=
## Anti-aliasing

All quadrature rules assume that the kernel varies smoothly across a cell. This assumption breaks down once the kernel support becomes small compared to the cell spacing $s$: the fixed nodes may then miss the support entirely, causing severe aliasing. The figure below shows a one-dimensional example for a Lucy kernel, with grid cell boundaries as grey vertical lines. On the left, the kernel is wide compared to the cell spacing ($\eta / \eta_{\rm crit} > 1$), so the quadrature nodes sample it densely and the cell integrals are accurate. On the right, the same kernel is narrower than a single cell ($\eta / \eta_{\rm crit} < 1$): its support can fall entirely between the fixed quadrature nodes, so the deposited weight depends on where the particle sits within its cell.

```{image} ../src/kernel_anti_alias.png
:alt: Anti-aliasing
:width: 70%
:align: center
```

For every particle $i$, `smudgy` therefore computes the ratio

$$
\frac{\eta_i}{\eta_{\rm crit}} = \min_a \frac{n \times 2\,\ell_{i,a}}{s_a}
$$

of the particle's kernel *diameter* to the cell spacing, where $n$ is the kernel's support multiple, $s_a$ is the cell spacing along axis $a$, and $\ell_{i,a}$ is the half-width of the unit kernel ellipsoid along that axis, i.e. $\ell_{i,a}=h_i$ for isotropic kernels and $\ell_{i,a} = [(\mathbfss{H}_i^2)_{aa}]^{1/2}$ for covariant ones. The safety parameter $\eta_{\rm crit}$ (`eta_crit` in {py:meth}`~smudgy.pointcloud.PointCloud.deposit`, default `4.0`) sets the minimum number of cells the kernel must span:

+ If $\eta_i \ge \eta_{\rm crit}$, the kernel covers several cells and its integral is estimated via numerical quadrature.
+ If $\eta_i < \eta_{\rm crit}$, the kernel is represented by a pre-computed, dimensionless sample table (shown below), which is rescaled by the particle's smoothing length or mapped through its smoothing tensor. Each sample then deposits its integral weight into the cell it falls into. Degenerate tensors with a vanishing axis always take this path, as do strongly elongated covariant kernels that span several cells along every grid axis but are so thin across their minor axis that no quadrature node falls inside their support. Its cost is independent of how small the kernel is relative to the grid.

```{image} ../src/kernel_sample_grid_2d.png
:alt: Pre-computed kernel sample table
:width: 70%
:align: center
```

The table is built once per kernel in the dimensionless coordinate $q = r/h$ and shared by all particles below the threshold. The support $q \in [0, n]$ is split into $n_q$ radial shells, each populated with samples distributed quasi-uniformly in angle (2D) or on the sphere via a Fibonacci spiral (3D). The exact integral over each shell, obtained from the analytic radial antiderivative, is divided evenly among its samples, so that the table integrates to unity by construction. The figure above shows the two-dimensional table for the Lucy kernel ($n = 1$), where the number of angular bins grows with radius, keeping the sample spacing roughly constant, and the shading shows the per-bin integral weight.

The table resolution follows from the sampling theorem: the sample spacing must not exceed half a cell width in the worst case, $\eta_i \to \eta_{\mathrm{crit}}^-$. This requires

$$
n_q = \left\lceil \frac{\eta_{\mathrm{crit}}}{2}\, n_{\mathrm{Ny}} \right\rceil = \lceil \eta_{\mathrm{crit}} \rceil,
$$

with Nyquist factor $n_{\mathrm{Ny}} = 2$. Applying the same argument to each shell's circumference (2D) or surface (3D) sets its number of samples. The number of samples is therefore not a separate user-facing parameter: since the table is Nyquist-safe in the worst case, every particle deposited through it is free of aliasing for any value of `eta_crit`. As the total sample count scales as $\eta_{\mathrm{crit}}^{d}$, `eta_crit` should be the smallest value that still ensures reliable quadrature above the threshold.